# 02. Baseline 모델 학습 및 오분류 원인 탐색

Train 내부에서만 학습·평가합니다. **Test 데이터는 읽지 않습니다.** 입력: `outputs/` 디렉터리의 전처리 산출물.

주요 지표: Macro F1, 암종별 Precision/Recall/F1, 변이 부담 그룹별 오류율, 주요 혼동 암종 쌍.

In [ ]:
%pip install numpy pandas scipy scikit-learn matplotlib seaborn

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
from scipy import sparse
from sklearn.naive_bayes import ComplementNB
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import f1_score, classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
from collections import Counter

ROOT = Path.cwd()
DATA_DIR = ROOT / 'outputs'
if not (DATA_DIR / 'X_train_binary.npz').exists():
    DATA_DIR = ROOT / 'outputs_review' / 'outputs'
assert (DATA_DIR / 'X_train_binary.npz').exists(), 'outputs 폴더의 전처리 파일을 확인하세요.'
REPORT_DIR = ROOT / 'baseline_reports'
REPORT_DIR.mkdir(exist_ok=True)
print('전처리 경로:', DATA_DIR)


In [ ]:
X = sparse.load_npz(DATA_DIR / 'X_train_binary.npz').tocsr()
y = np.load(DATA_DIR / 'y_train.npy', allow_pickle=True).astype(str)
s = np.load(DATA_DIR / 'split_indices.npz')
fit_idx, valid_idx = s['fit_idx'], s['valid_idx']
meta = pd.read_csv(DATA_DIR / 'train_patient_summary.csv')
assert len(y) == X.shape[0] == len(meta)
assert set(fit_idx).isdisjoint(set(valid_idx))
assert len(set(y[fit_idx])) == len(set(y[valid_idx])) == 26
print('행렬:',X.shape,'학습/검증:',len(fit_idx),len(valid_idx))


## 1. 기본 모델 비교
표본 불균형 대응 효과를 비교하기 위해 Complement Naive Bayes와 클래스 가중 SGD(Log Loss)를 사용합니다. 같은 분할에서 평가합니다.

In [ ]:
models = {
    'ComplementNB': ComplementNB(alpha=1.0),
    'SGD_LogLoss_Balanced': SGDClassifier(loss='log_loss', class_weight='balanced', random_state=42, max_iter=1500, tol=1e-3)
}
results, predictions = [], {}
for name, model in models.items():
    model.fit(X[fit_idx], y[fit_idx])
    pred = model.predict(X[valid_idx])
    predictions[name] = pred
    results.append({'model':name, 'macro_f1':f1_score(y[valid_idx],pred,average='macro'),
                    'weighted_f1':f1_score(y[valid_idx],pred,average='weighted'),
                    'accuracy':accuracy_score(y[valid_idx],pred)})
    report = pd.DataFrame(classification_report(y[valid_idx],pred,output_dict=True,zero_division=0)).T
    report.to_csv(REPORT_DIR/f'class_report_{name}.csv',encoding='utf-8-sig')
comparison = pd.DataFrame(results).sort_values('macro_f1',ascending=False)
comparison.to_csv(REPORT_DIR/'baseline_comparison.csv',index=False,encoding='utf-8-sig')
display(comparison)


## 2. 암종별 F1 및 표본 수 비교
표본 수가 적은 암종에서 실제로 성능이 낮은지 분석합니다. **데이터 내 소수 클래스가 임상적으로 희귀 암종이라는 뜻은 아닙니다.**

In [ ]:
best_name = comparison.iloc[0]['model']
pred = predictions[best_name]
rep = pd.DataFrame(classification_report(y[valid_idx],pred,output_dict=True,zero_division=0)).T
counts = pd.Series(y, name='SUBCLASS').value_counts()
per_class = rep.loc[sorted(counts.index),['precision','recall','f1-score','support']].copy()
per_class['train_total'] = per_class.index.map(counts)
per_class.sort_values('f1-score').to_csv(REPORT_DIR/'per_class_sorted.csv',encoding='utf-8-sig')
display(per_class.sort_values('f1-score').head(12))


## 3. 혼동되는 암종 쌍 찾기
모델이 유사한 암종을 잘못 예측하는지 파악합니다.

In [ ]:
labels = sorted(np.unique(y))
cm = confusion_matrix(y[valid_idx],pred,labels=labels)
confusions = []
for i,truth in enumerate(labels):
    for j,guess in enumerate(labels):
        if i != j and cm[i,j] > 0:
            confusions.append({'true':truth,'predicted':guess,'n':int(cm[i,j]),'true_support':int(cm[i].sum())})
confusions = pd.DataFrame(confusions).sort_values('n',ascending=False)
confusions.to_csv(REPORT_DIR/'confused_pairs.csv',index=False,encoding='utf-8-sig')
display(confusions.head(15))
fig,ax=plt.subplots(figsize=(12,10))
im=ax.imshow(cm,cmap='Blues')
ax.set_xticks(range(len(labels)),labels,rotation=90)
ax.set_yticks(range(len(labels)),labels)
ax.set_xlabel('Predicted');ax.set_ylabel('True');ax.set_title(f'Confusion matrix: {best_name}')
fig.colorbar(im,ax=ax);plt.tight_layout();plt.show()


## 4. 변이 부담 그룹별 오류율
전처리 산출물의 low/middle/high 그룹을 활용하여 성능과 오류를 비교합니다. 그룹은 **이 노트북에서 모델 선택에 사용하지 않고 탐색적 분석**에만 사용합니다.

In [ ]:
val = meta.iloc[valid_idx].copy().reset_index(drop=True)
val['true'] = y[valid_idx]
val['pred'] = pred
val['correct'] = val['true']==val['pred']
groups = val.groupby('mutation_group',dropna=False).agg(n=('correct','size'),accuracy=('correct','mean'),avg_mutations=('mutation_count','mean')).reset_index()
groups['error_rate'] = 1-groups['accuracy']
groups.to_csv(REPORT_DIR/'burden_group.csv',index=False,encoding='utf-8-sig')
display(groups)
val.to_csv(REPORT_DIR/'validation_predictions.csv',index=False,encoding='utf-8-sig')


## 5. 연구 기록
결과를 보고 다음을 기록하세요.

1. **어느 암종의 F1이 낮은가?** 표본 수가 원인이라고 단정하지 말고 함께 비교.
2. **어떤 암종 쌍이 자주 혼동되는가?** 실제 유전자 분포 비교는 학습 폴드에서 수행.
3. **저변이/고변이 환자군의 오류가 높은가?** 암종 구성 차이를 통제해야 함.
4. **다음 실험**은 하나의 가설만 바꾸고 같은 검증 분할에서 비교. 여러 실험으로 튜닝할 경우 별도 교차검증·최종 홀드아웃을 추가.

주의: Test 데이터, Test 통계량, Test 예측 분포, Public 점수를 모델 또는 전처리 선택에 활용하지 않습니다.